# HW6 : 총액 regression model 을 고르고, API 로 내보낸다
21011658 김영헌. `sales_clean.csv` (HW5 결과) 를 이 폴더에 두고 Restart → Run All.

In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

## ① 읽고 나눈다 : X = 수량 · 단가, y = 총액

In [2]:
sales = pd.read_csv("sales_clean.csv")
X = sales[["quantity", "unit_price"]]
y = sales["total_sales"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)
print("training :", len(X_train), " test :", len(X_test))

training : 803  test : 201


## ② model 셋의 test R²

In [3]:
X2_train = X_train.assign(qp=X_train["quantity"] * X_train["unit_price"])   # 수량 × 단가 칸을 더함
X2_test  = X_test.assign(qp=X_test["quantity"] * X_test["unit_price"])
lin  = LinearRegression().fit(X_train, y_train)
lin2 = LinearRegression().fit(X2_train, y_train)
tree = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)
print("linear regression             test R² :", round(lin.score(X_test, y_test), 3))
print("linear regression + qp        test R² :", round(lin2.score(X2_test, y_test), 3))
print("decision tree (no limit)      test R² :", round(tree.score(X_test, y_test), 3))

linear regression             test R² : 0.862
linear regression + qp        test R² : 1.0
decision tree (no limit)      test R² : 0.994


## ③ linear regression 이 0.862 에 그친 이유, qp 를 더하면 1.0 이 되는 이유

In [4]:
print("linear      coef :", lin.coef_.round(1), " intercept :", round(lin.intercept_, 1))
print("linear + qp coef :", lin2.coef_.round(3), " intercept :", round(lin2.intercept_, 1))

linear      coef : [2.515409e+05 4.900000e+01]  intercept : -12387684.5
linear + qp coef : [0. 0. 1.]  intercept : -0.0


- linear 의 규칙은 `a × 수량 + b × 단가 + c` 모양이라 두 열을 곱한 항이 없음. 실제 규칙은 `총액 = 수량 × 단가` 라서 곱의 모양을 못 그려 0.862 에 멈춤
- qp 칸을 주면 qp 의 계수 1, 나머지 0, 절편 0 으로 실제 규칙과 같아져서 1.0

## ④ max_depth 별 training · test R²

In [5]:
rows = []
for dep in [1, 2, 3, 5, 8, 12, None]:
    m = DecisionTreeRegressor(max_depth=dep, random_state=0).fit(X_train, y_train)
    rows.append([str(dep), m.get_n_leaves(),
                 round(m.score(X_train, y_train), 3), round(m.score(X_test, y_test), 3)])
pd.DataFrame(rows, columns=["max_depth", "leaves", "training R²", "test R²"])

,max_depth,leaves,training R²,test R²
0,1,2,0.361,0.346
1,2,4,0.739,0.711
2,3,8,0.876,0.846
3,5,32,0.972,0.961
4,8,250,0.998,0.991
5,12,781,1.000,0.994
6,None,800,1.000,0.994


## ⑤ cross-validation (cv=5) 으로 max_depth 를 고른다

In [6]:
for dep in [3, 5, 8, 12, None]:
    s = cross_val_score(DecisionTreeRegressor(max_depth=dep, random_state=0), X, y, cv=5)
    print(f"max_depth {str(dep):4s}  mean {s.mean():.3f}  std {s.std():.3f}")

max_depth 3     mean 0.845  std 0.019
max_depth 5     mean 0.961  std 0.003


max_depth 8     mean 0.992  std 0.000
max_depth 12    mean 0.995  std 0.000


max_depth None  mean 0.995  std 0.001


- **max_depth 8** 을 고름. 평균 0.992 · std 0.000 이고 그 뒤 (12 · None) 로는 0.003 차이뿐인데 잎은 250 개로 781 · 800 개의 1/3. 거의 같으면 단순한 쪽
- training 1.000 · test 0.994 로 차이가 작은 것은 `총액 = 수량 × 단가` 라는 규칙이 데이터에 실제로 있어서, 잘게 나눠 외워도 test 에서 그대로 맞기 때문
- None 의 평균 0.995 가 수업 자료 (0.994) 와 다른 것 : HW5 에서 직접 만든 `sales_clean.csv` 의 총액 131 줄이 수업 자료 파일과 0.00000001 원 미만 (float 저장 자리) 으로 달라 제한 없는 tree 의 잎 나눔이 조금 바뀜. 평균 0.9946 vs 0.9944 로 셋째 자리 반올림만 갈림

## ⑥ 고른 model 을 저장한다 : linear + qp

In [7]:
joblib.dump(lin2, "sales_model.pkl")
model = joblib.load("sales_model.pkl")
x = pd.DataFrame([[40, 200000]], columns=["quantity", "unit_price"])
x["qp"] = x["quantity"] * x["unit_price"]   # app.py 와 같은 이름 · 순서
print("linear + qp       ->", int(round(model.predict(x)[0])))

linear + qp       -> 8000000


## ⑦ 왜 그 model 인가 : 수량 40 · 단가 200,000

In [8]:
tree8 = DecisionTreeRegressor(max_depth=8, random_state=0).fit(X_train, y_train)
print("linear + qp       ->", int(round(model.predict(x)[0])))
print("tree (depth 8)    ->", int(round(tree8.predict(x[["quantity", "unit_price"]])[0])))
print("quantity × price  ->", 40 * 200000)

linear + qp       -> 8000000
tree (depth 8)    -> 7362301
quantity × price  -> 8000000


- linear + qp 8,000,000 · tree (depth 8) 7,362,301 · 실제 곱 8,000,000
- **linear + qp** 를 고름. 실제 곱과 한 원도 다르지 않음. tree 는 잎에 모인 training 줄들의 평균을 내놓는 계단 모양이라 637,699 원 (약 8 %) 모자람
- 공식이 있으니 서버에서 곱만 해도 같은 답이지만, model 을 API 로 내보내는 과제라 그 공식을 그대로 배운 linear + qp 를 올림